# HTMX v4 Support for FastHTML

**Objective:** Add htmx v4 compatibility to FastHTML.

**Approach:**
- Create v4-compatible versions of affected functions, using a `4` suffix (e.g., `func4`)
- Add an `htmx4: bool` parameter to `FastHTML.__init__` to toggle between versions
- Default to `False` initially for backward compatibility; switch default in a future release

This project includes a patch package `hx4_patch`. Running `from hx4_patch.core import *` should perform all necessary setup for htmx4.

The source code is in `/app/data/fasthtml-example/hx4_patch/hx4_patch/core.py`

## Testing

We test by reimplementing all the examples inside solveit, using htmx v4. To make FastHTML work in solveit, we need to run `from fasthtml.jupyter import *` and `srv = JupyUvi(app)`. We can also export the dialog to .py and use 'IN_SOLVEIT' env variable to decide to use `serve` or `JupyUvi`

When creating a dialog based on an existing .py example, keep the same code structure but make these changes for htmx v4:
1. Add `from hx4_patch.core import *` after other imports
2. Use `FastHTML` or `fast_app` with `htmx4=True`, `hxmx=False`
3. Replace `serve()` with `JupyUvi(app)`

# HTMX v4 guide

+++
title = "Changes in htmx 4.0"
[extra]
table_of_contents = true
+++

htmx 4.0 is a ground up rewrite of the implementation of htmx, using the `fetch()` API.  This document outlines the 
major changes between htmx 2.x and htmx 4.x.

## Major Changes

### fetch() API replaces XMLHttpRequest
- All AJAX requests now use the native fetch() API instead of XMLHttpRequest
- Enables streaming response support
- Simplifies implementation of htmx significantly

### Explicit Attribute Inheritance
- Attribute inheritance is now explicit by default, using the `:inherited` modifier
- By default, in htmx 4.x you now use `hx-attribute:inherited="value"` syntax to inherit an attribute
- This applies to all inheritable attributes: `hx-boost:inherited`, `hx-target:inherited`, `hx-confirm:inherited`, etc.
- This improves locality of behavior by making inheritance explicit
- You can revert to implicit inheritance by setting `htmx.config.implicitInheritance` to `true`

### Event Naming Convention Changed
- New event naming convention: `htmx:phase:action[:sub-action]` (colon-separated)
- Many event names have changed (See the [migration guide](/migration-guide-htmx-4#event-changes))
- This provides more consistent & predictable event naming

### History Storage
- History no longer uses `localStorage` to store snapshots of previous pages
- History now issues a full page refresh request on history navigation
- This is a much, much more reliable history restoration mechanic
- We will be creating a caching history extension for people that want the old behavior

### Non-200 Swapping Defaults
- In htmx 2.0, responses with `4xx` and `5xx` response codes did not swap by default
- In htmx 4.0, all responses will swap except for [`204 - No Content`](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/204)
  and [`304 - Not Modified`](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/304)
- You can revert to not swapping `4xx` and `5xx` responses by setting `htmx.config.noSwap` to `[204, 304, '4xx', '5xx']`

## New Features

### Morphing Swap
- htmx now ships with morph swap styles are now available, based on the original `idiomorph` algorithm
- `innerMorph` - morphs the children of the target element
- `outerMorph` - morphs the target element itself
- Does a better job of preserving local state when targeting large DOM trees

### Built-in Streaming Response Support
- Streaming functionality/SSE now built into core htmx
- Improved event handling and reconnection logic 
- Configure globally using `<meta name="htmx-config">`
   ```html
   <!-- Global defaults -->
   <meta name="htmx-config" content="{
       streams:{
         reconnect: false,
         reconnectDelay: 500,
         reconnectMaxDelay: 60000,
         reconnectMaxAttempts: 10,
         reconnectJitter: 0.3,
         pauseInBackground: false
       }
   }">
   ```
- Or per-element using `hx-config` attribute
  ```html
  <!-- Overrides global default -->
  <div hx-get="/events" 
       hx-trigger="load"
       hx-config="{stream: {reconnect: true}}"
  ```

### View Transitions
- View Transitions API enabled by default (maybe not!)
- Provides smooth animated transitions between DOM states
- Set `htmx.config.transitions = false` to disable

### Scripting API
- New unified scripting API for async operations
- Better integration points for custom JavaScript
- Improved support for async/await patterns

### Unified Request Context
- All events now provide consistent `ctx` object
- Easier to access request/response information
- More predictable event handling

### Modern Swap Terminology
- New modern swap style names supported alongside classic names
- `before` (equivalent to `beforebegin`)
- `after` (equivalent to `afterend`)
- `prepend` (equivalent to `afterbegin`)
- `append` (equivalent to `beforeend`)
- Both old and new terminology work (backward compatible)
- Example: `hx-swap="prepend"` works the same as `hx-swap="afterbegin"`

### Inheritance Attribute Modifiers
- New `:append` modifier for attributes to append values to inherited values
- Values are comma-separated when appended
- Example: `hx-include:append=".child"` appends `.child` to any inherited `hx-include` value
- Can be combined with `:inherited` for chaining: `hx-include:inherited:append=".parent"`
- Works with all htmx attributes that accept value lists

### HTTP Status Code Conditional Swapping
- New `hx-status:XXX` attribute pattern for status-specific swap behaviors
- Allows different swap strategies based on HTTP response status
- Supports exact codes: `hx-status:404="none"`
- Supports wildcards: `hx-status:2xx="innerHTML"`, `hx-status:5xx="#error"`
- Example: `hx-status:404="#not-found"` swaps into different target on 404
- Overrides default swap behavior when status code matches

### Partial Tags
- New `<hx-partial>` tag for multiple targeted swaps in one response
- Provides explicit control over swap targets via `hx-target` attribute
- Alternative to out-of-band swaps when you want explicit targeting
- Example:
  ```html
  <hx-partial hx-target="#messages" hx-swap="beforeend">
    <div>New message</div>
  </hx-partial>
  <hx-partial hx-target="#notifications">
    <span class="badge">5</span>
  </hx-partial>
  ```
- Each partial specifies its own target and swap strategy
- More explicit than OOB swaps which rely on matching `id` attributes

## Attribute Changes

### Renamed Attributes
- `hx-disable` renamed to `hx-ignore`
- `hx-disabled-elt` renamed to `hx-disable` :/

### Removed Attributes
- `hx-vars` - use `hx-vals` with `js:` prefix instead
- `hx-params` - use `htmx:config:request` event to filter parameters
- `hx-prompt` - use `hx-confirm` with async JavaScript function
- `hx-ext` - extensions now work via event listeners
- `hx-disinherit` - no longer needed (inheritance is explicit)
- `hx-inherit` - no longer needed (inheritance is explicit)
- `hx-request` - use `hx-config` instead
- `hx-history` - removed (history no longer uses local storage)
- `hx-history-elt` - removed (history uses target element)

### New Attributes
- `hx-action` - specifies URL for requests (use with `hx-method`)
- `hx-method` - specifies HTTP method (use with `hx-action`)
- `hx-config` - configure request behavior using JSON
- `hx-status:XXX` - conditional swap behavior based on HTTP status code (e.g., `hx-status:404="none"`)
- `hx-ignore` - replaces htmx 2.x `hx-disable` for disabling htmx processing

### Attribute Modifier Syntax
- `:inherited` - explicitly inherit attribute value from parent (e.g., `hx-target:inherited="this"`)
- `:append` - append value to inherited value (e.g., `hx-include:append=".child"`)
- `:inherited:append` - combine inheritance and appending (e.g., `hx-vals:inherited:append='{"key":"value"}'`)

## Event Changes

### Event Name Mappings
- `htmx:afterOnLoad` → `htmx:after:init`
- `htmx:afterProcessNode` → `htmx:after:init`
- `htmx:afterRequest` → `htmx:after:request`
- `htmx:afterSettle` → `htmx:after:swap`
- `htmx:afterSwap` → `htmx:after:swap`
- `htmx:beforeCleanupElement` → `htmx:before:cleanup`
- `htmx:beforeHistorySave` → `htmx:before:history:update`
- `htmx:beforeOnLoad` → `htmx:before:init`
- `htmx:beforeProcessNode` → `htmx:before:init`
- `htmx:beforeRequest` → `htmx:before:request`
- `htmx:beforeSwap` → `htmx:before:swap`
- `htmx:configRequest` → `htmx:config:request`
- `htmx:historyCacheMiss` → `htmx:before:restore:history`
- `htmx:historyRestore` → `htmx:after:restore:history`
- `htmx:load` → `htmx:after:init`
- `htmx:oobAfterSwap` → `htmx:after:oob:swap`
- `htmx:oobBeforeSwap` → `htmx:before:oob:swap`
- `htmx:pushedIntoHistory` → `htmx:after:push:into:history`
- `htmx:replacedInHistory` → `htmx:after:replace:into:history`
- `htmx:responseError` → `htmx:error`
- `htmx:sendError` → `htmx:error`
- `htmx:swapError` → `htmx:error`
- `htmx:targetError` → `htmx:error`
- `htmx:timeout` → `htmx:error`

### New Events
- `htmx:after:cleanup` - fires after element cleanup completes
- `htmx:after:history:update` - fires after history state is updated
- `htmx:after:process` - fires after element processing completes
- `htmx:before:settle` - fires before settle phase begins
- `htmx:after:settle` - fires after settle phase completes
- `htmx:finally:request` - fires in finally block after request (success or error)
- `htmx:before:sse:stream` - fires before SSE stream begins
- `htmx:after:sse:stream` - fires after SSE stream ends
- `htmx:before:sse:message` - fires before processing SSE message
- `htmx:after:sse:message` - fires after processing SSE message
- `htmx:before:sse:reconnect` - fires before attempting SSE reconnection
- `htmx:before:viewTransition` - fires before view transition starts
- `htmx:after:viewTransition` - fires after view transition completes

### Extensions Are Now Globally Registered
- Extensions no longer require an explicit `hx-ext` attribute
- Simpler extension architecture

# WebSocket

## Websocket v4 doc
+++
title = "htmx WebSocket Extension"
+++

## Usage

Use these attributes to configure WebSocket behavior:

| Attribute | Description |
|-----------|-------------|
| `hx-ws:connect="<url>"` | Establishes a WebSocket connection to the specified URL |
| `hx-ws:send` | Sends form data or `hx-vals` to the WebSocket on trigger |
| `hx-ws:send="<url>"` | Like `hx-ws:send` but creates its own connection to the URL |

**JSX-Compatible Variants:** For frameworks that don't support colons in attribute names, use hyphen variants: `hx-ws-connect` and `hx-ws-send`.

### Basic Example

```html
<div hx-ws:connect="/chatroom" hx-target="#messages" hx-swap="beforeend">
    <div id="messages"></div>
    <form hx-ws:send>
        <input name="message" placeholder="Type a message...">
        <button type="submit">Send</button>
    </form>
</div>
```

This example:
1. Establishes a WebSocket connection to `/chatroom` when the page loads
2. Appends incoming HTML messages to `#messages`
3. Sends form data as JSON when the form is submitted

## URL Normalization

WebSocket URLs are automatically normalized:

| Input | Output (on HTTPS page) |
|-------|------------------------|
| `/ws/chat` | `wss://example.com/ws/chat` |
| `ws://localhost:8080/ws` | `ws://localhost:8080/ws` |
| `https://api.example.com/ws` | `wss://api.example.com/ws` |
| `//cdn.example.com/ws` | `wss://cdn.example.com/ws` |

This means you can use simple relative paths in most cases, and the extension will construct the correct WebSocket URL.

## Receiving Messages

### JSON Envelope Format

Messages from the server should be JSON objects:

```json
{
    "channel": "ui",
    "format": "html",
    "target": "#notifications",
    "swap": "beforeend",
    "payload": "<div class='notification'>New message!</div>",
    "request_id": "abc-123"
}
```

| Field | Default | Description |
|-------|---------|-------------|
| `channel` | `"ui"` | Message routing channel |
| `format` | `"html"` | Content format |
| `target` | Element's `hx-target` | CSS selector for target element |
| `swap` | Element's `hx-swap` | Swap strategy (innerHTML, beforeend, etc.) |
| `payload` | — | The content to swap |
| `request_id` | — | Matches response to original request |

**Minimal Example** (using all defaults):
```json
{"payload": "<div>Hello World</div>"}
```

### Channels

- **`ui` channel** (default): HTML content is swapped into the target element using htmx's swap pipeline
- **Custom channels**: Emit an `htmx:wsMessage` event for application handling

```javascript
// Handle custom channel messages
document.addEventListener('htmx:wsMessage', (e) => {
    if (e.detail.channel === 'notifications') {
        showNotification(e.detail.payload);
    }
});
```

### Request-Response Matching

The extension generates a unique `request_id` for each message sent. When the server includes this `request_id` in the response:

- Content is swapped into the element that originated the request
- That element's `hx-target` and `hx-swap` attributes are respected
- Enables request-response patterns over WebSocket

### Legacy Format (Deprecated)

For backward compatibility, the extension also supports `<hx-partial>` elements:

```html
<hx-partial id="notifications">
    <div>New notification</div>
</hx-partial>
```

## Sending Messages

When an element with `hx-ws:send` is triggered, the extension sends a JSON message:

```json
{
    "type": "request",
    "request_id": "550e8400-e29b-41d4-a716-446655440000",
    "event": "submit",
    "headers": {
        "HX-Request": "true",
        "HX-Current-URL": "https://example.com/chat",
        "HX-Trigger": "chat-form",
        "HX-Target": "#messages"
    },
    "values": {
        "message": "Hello!",
        "tags": ["urgent", "public"]
    },
    "path": "wss://example.com/chatroom",
    "id": "chat-form"
}
```

| Field | Description |
|-------|-------------|
| `type` | Always `"request"` for client-to-server messages |
| `request_id` | Unique ID for request/response matching |
| `event` | DOM event type that triggered the send (e.g., `"submit"`, `"click"`) |
| `headers` | HTMX-style headers for server-side routing |
| `values` | Form data and `hx-vals` (multi-value fields preserved as arrays) |
| `path` | The normalized WebSocket URL |
| `id` | Element ID (only present if element has an `id`) |

### Forms

```html
<form hx-ws:send>
    <input name="username">
    <input name="message">
    <button type="submit">Send</button>
</form>
```

Form data is collected and sent as the `values` object. Multi-value fields (like checkboxes or multi-selects) are preserved as arrays.

### Buttons with hx-vals

```html
<button hx-ws:send hx-vals='{"action": "increment"}'>+1</button>
```

### Modifying Messages Before Send

Use the `htmx:before:ws:send` event to modify or cancel messages:

```javascript
document.addEventListener('htmx:before:ws:send', (e) => {
    // Add authentication token
    e.detail.data.headers['Authorization'] = 'Bearer ' + getToken();
    
    // Or cancel the send
    if (!isValid(e.detail.data)) {
        e.preventDefault();
    }
});
```

## Configuration

Configure the extension via `htmx.config.websockets`:

```javascript
htmx.config.websockets = {
    reconnect: true,           // Enable auto-reconnect (default: true)
    reconnectDelay: 1000,      // Initial reconnect delay in ms (default: 1000)
    reconnectMaxDelay: 30000,  // Maximum reconnect delay in ms (default: 30000)
    reconnectJitter: true,     // Add randomization to delays (default: true)
    pendingRequestTTL: 30000   // Time-to-live for pending requests in ms (default: 30000)
};
```

### Reconnection Strategy

The extension uses exponential backoff with optional jitter:

- **Base formula**: `delay = min(reconnectDelay × 2^(attempts-1), reconnectMaxDelay)`
- **Jitter**: Adds ±25% randomization to avoid thundering herd
- **Reset**: Attempts counter resets to 0 on successful connection

Example reconnection delays with defaults:
- Attempt 1: ~1000ms
- Attempt 2: ~2000ms
- Attempt 3: ~4000ms
- Attempt 4: ~8000ms
- Attempt 5+: ~30000ms (capped)

### Connection Triggers

By default, connections are established immediately when the element is processed:

```html
<!-- Connects immediately when element appears (default) -->
<div hx-ws:connect="/ws">

<!-- Explicit load trigger - same behavior as no trigger -->
<div hx-ws:connect="/ws" hx-trigger="load">

<!-- Deferred connection - only connects when button is clicked -->
<div hx-ws:connect="/ws" hx-trigger="click from:#connect-btn">
```

Use `hx-trigger` when you want to **delay** connection establishment (e.g., wait for user action).

**Note:** Only bare event names are supported for connection triggers. Modifiers like `delay`, `throttle`, `once` are **not supported**. For complex connection logic, use the `htmx:before:ws:connect` event.

## Events

### Connection Lifecycle

| Event | Cancelable | Detail | Description |
|-------|------------|--------|-------------|
| `htmx:before:ws:connect` | ✅ | `{url}` | Before establishing connection |
| `htmx:after:ws:connect` | ❌ | `{url, socket}` | After successful connection |
| `htmx:ws:close` | ❌ | `{url, code, reason}` | When connection closes |
| `htmx:ws:error` | ❌ | `{url, error}` | On connection error |
| `htmx:ws:reconnect` | ❌ | `{url, attempts}` | Before reconnection attempt |

### Message Events

| Event | Cancelable | Detail | Description |
|-------|------------|--------|-------------|
| `htmx:before:ws:send` | ✅ | `{data, element, url}` | Before sending (data is modifiable) |
| `htmx:after:ws:send` | ❌ | `{data, url}` | After message sent |
| `htmx:wsSendError` | ❌ | `{element, error}` | When send fails |
| `htmx:before:ws:message` | ✅ | `{envelope, element}` | Before processing received message |
| `htmx:after:ws:message` | ❌ | `{envelope, element}` | After processing received message |
| `htmx:wsMessage` | ❌ | `{channel, format, payload, ...}` | For non-UI channel messages |
| `htmx:wsUnknownMessage` | ❌ | `{data, parseError}` | For non-JSON messages |

### Event Examples

**Cancel Connection Based on Condition:**
```javascript
document.addEventListener('htmx:before:ws:connect', (e) => {
    if (document.hidden) {
        e.preventDefault(); // Don't connect in background tab
    }
});
```

**Handle Custom Messages:**
```javascript
document.addEventListener('htmx:wsMessage', (e) => {
    if (e.detail.channel === 'audio') {
        playAudioNotification(e.detail.payload);
    }
});
```

**Log All WebSocket Activity:**
```javascript
document.addEventListener('htmx:after:ws:connect', (e) => {
    console.log('Connected to', e.detail.url);
});
document.addEventListener('htmx:ws:close', (e) => {
    console.log('Disconnected from', e.detail.url, 'code:', e.detail.code);
});
```

## Connection Management

### Reference Counting

Multiple elements can share a single WebSocket connection:

```html
<div hx-ws:connect="/notifications" id="notif-1">
    <!-- Uses connection to /notifications -->
</div>
<div hx-ws:connect="/notifications" id="notif-2">
    <!-- Shares the same connection -->
</div>
```

When all elements using a connection are removed from the DOM, the connection is automatically closed.

### Element Cleanup

When elements are removed (e.g., via htmx swap), the extension:
1. Decrements the reference count for the connection
2. Removes event listeners from the element
3. Closes the WebSocket if no elements remain

This happens automatically through htmx's element cleanup lifecycle.

## HTML Swapping

When a `channel: "ui"` message arrives, the extension uses htmx's internal `insertContent` API, which provides:

- All swap styles (`innerHTML`, `outerHTML`, `beforebegin`, `afterend`, `beforeend`, `afterbegin`, `delete`, `none`)
- Preserved elements (`hx-preserve`)
- Auto-focus handling
- Scroll handling  
- Proper cleanup of removed elements
- `htmx.process()` called on newly inserted content

### Target Resolution

Target is determined in this order:
1. `target` field in the message envelope
2. `hx-target` attribute on the element that sent the request (if `request_id` matches)
3. `hx-target` attribute on the connection element
4. The connection element itself

### Swap Strategy

Swap strategy is determined in this order:
1. `swap` field in the message envelope
2. `hx-swap` attribute on the target element
3. `htmx.config.defaultSwap` (default: `innerHTML`)

## Examples

### Live Chat

```html
<div hx-ws:connect="/chat">
    <div id="messages" hx-target="this" hx-swap="beforeend"></div>
    <form hx-ws:send>
        <input name="message" placeholder="Message..." autocomplete="off">
        <button type="submit">Send</button>
    </form>
</div>
```

Server sends:
```json
{"payload": "<div class='message'><b>User:</b> Hello!</div>"}
```

### Real-Time Notifications

```html
<div hx-ws:connect="/notifications"
     hx-target="#notification-list"
     hx-swap="afterbegin">
    <div id="notification-list"></div>
</div>
```

### Interactive Counter

```html
<div hx-ws:connect="/counter">
    <div id="count" hx-target="this">0</div>
    <button hx-ws:send hx-vals='{"action":"increment"}'>+</button>
    <button hx-ws:send hx-vals='{"action":"decrement"}'>-</button>
</div>
```

### Multiple Widgets Sharing Connection

```html
<div hx-ws:connect="/dashboard">
    <div id="cpu-usage">--</div>
    <div id="memory-usage">--</div>
    <div id="disk-usage">--</div>
</div>
```

Server sends targeted updates:
```json
{"target": "#cpu-usage", "payload": "<span>45%</span>"}
{"target": "#memory-usage", "payload": "<span>2.3 GB</span>"}
```

## Migrating from Previous Versions

### Attribute Changes

| Old (htmx 2.x) | New (htmx 4.x) | Notes |
|----------------|----------------|-------|
| `ws-connect="<url>"` | `hx-ws:connect="<url>"` | Or `hx-ws-connect` for JSX |
| `ws-send` | `hx-ws:send` | Or `hx-ws-send` for JSX |
| `hx-ext="ws"` | Not required | Extension auto-registers when loaded |

The old `ws-connect` and `ws-send` attributes still work but emit a deprecation warning.

### Event Changes

| Old Event | New Event | Notes |
|-----------|-----------|-------|
| `htmx:wsConnecting` | — | Removed |
| `htmx:wsOpen` | `htmx:after:ws:connect` | Different detail structure |
| `htmx:wsClose` | `htmx:ws:close` | Now includes `code` and `reason` |
| `htmx:wsError` | `htmx:ws:error` | Similar |
| `htmx:wsBeforeMessage` | `htmx:before:ws:message` | Different detail structure |
| `htmx:wsAfterMessage` | `htmx:after:ws:message` | Different detail structure |
| `htmx:wsConfigSend` | `htmx:before:ws:send` | Modify `e.detail.data` instead |
| `htmx:wsBeforeSend` | `htmx:before:ws:send` | Combined into one event |
| `htmx:wsAfterSend` | `htmx:after:ws:send` | Similar |

### Configuration Changes

| Old | New |
|-----|-----|
| `htmx.config.wsReconnectDelay` | `htmx.config.websockets.reconnectDelay` |
| `createWebSocket` option | Not supported (use events) |
| `wsBinaryType` option | Not supported |

### Message Format Changes

**Send payload** now includes `type`, `request_id`, `event`, and structured `headers` object instead of `HEADERS` string.

**Receive format** now expects JSON envelope with `channel`, `format`, `target`, `swap`, `payload` fields instead of raw HTML or `hx-swap-oob`.

### Socket Wrapper Removed

The `socketWrapper` object is no longer exposed in events. Use the standard WebSocket events and the extension's event system instead.